### Data Preparation

**Goal**:
Clean and prepare data for analytical modeling and unit economics calculations.

**Key tasks**:
- Data cleaning
- Date conversion
- Handling missing values
- Creating marketing channels
- Preparing datasets for SQL layer

#### 1. Setup & Load Data

In [ ]:
# Set working directory
import local_config
from local_config import directory_path
import os
os.chdir(directory_path)

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# display settings
pd.set_option('display.max_columns', None)

# Load data
accounts = pd.read_csv('data/accounts.csv')
subscriptions = pd.read_csv('data/subscriptions.csv')
feature_usage = pd.read_csv('data/feature_usage.csv')
support_tickets = pd.read_csv('data/support_tickets.csv')
churn_events = pd.read_csv('data/churn_events.csv')

#### 2. Data Cleaning
2.1 Date conversion

In [ ]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'], errors='coerce')

subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'], errors='coerce')
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'], errors='coerce')

feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'], errors='coerce')

support_tickets['submitted_at'] = pd.to_datetime(support_tickets['submitted_at'], errors='coerce')
support_tickets['closed_at'] = pd.to_datetime(support_tickets['closed_at'], errors='coerce')

churn_events['churn_date'] = pd.to_datetime(churn_events['churn_date'], errors='coerce')

All date columns were converted to datetime format. Invalid values were coerced to NaT.

2.2 Handling missing values

In [ ]:
# satisfaction_score
support_tickets['satisfaction_score'] = support_tickets['satisfaction_score'].fillna(
    support_tickets['satisfaction_score'].mean()
)

# feedback_text
churn_events['feedback_text'] = churn_events['feedback_text'].fillna('n/a')

# active subscriptions
subscriptions['is_active'] = subscriptions['end_date'].isna()

- Missing satisfaction scores were imputed with the mean value
- Missing feedback text was replaced with 'n/a'
- Missing end_date is preserved as it indicates active subscriptions

2.3 Data consistency checks

In [ ]:
# end_date should be after start_date
subscriptions[
    subscriptions['end_date'] < subscriptions['start_date']
]

All `end_date` values are greater than `start_date`, which confirms data consistency.

#### 3. Marketing Channel Preparation

3.1 Mapping referral → channel

In [ ]:
accounts['channel'] = accounts['referral_source']

*Marketing Channel Assumption*

The dataset does not contain explicit marketing channel data. The `referral_source` field is used as a proxy for acquisition channels with the following mapping:

- organic → SEO / direct traffic  
- ads → paid marketing  
- partner → partnerships / affiliates  
- event → offline events / conferences  
- other → unknown / mixed sources  

This approach allows us to approximate CAC and analyze unit economics by channel.

3.2 Create synthetic marketing spend

In [ ]:
date_range = pd.date_range(start='2023-01-01', end='2024-12-31', freq='MS')

channels = accounts['channel'].unique()

marketing_spend = pd.DataFrame([
    {
        'date': date,
        'channel': channel,
        'spend': np.random.randint(5000, 20000)
    }
    for date in date_range
    for channel in channels
])

Synthetic marketing spend data was generated to enable CAC calculation.

3.3 Add variability

In [ ]:
channel_multiplier = {
    'ads': 1.5,
    'organic': 0.3,
    'partner': 1.2,
    'event': 1.0,
    'other': 0.8
}

marketing_spend['spend'] = marketing_spend.apply(
    lambda x: x['spend'] * channel_multiplier.get(x['channel'], 1),
    axis=1
)

3.4 CAC modeling

In [ ]:
# Users by channels and dates 
new_users = accounts.groupby(
    [accounts['signup_date'].dt.to_period('M'), 'channel']
)['account_id'].count().reset_index(name='new_users')

In [ ]:
# join users with spend
marketing_spend['date'] = marketing_spend['date'].dt.to_period('M')

cac_table = marketing_spend.merge(
    new_users,
    left_on=['date', 'channel'],
    right_on=['signup_date', 'channel'],
    how='left'
)

In [ ]:
cac_table['new_users'] = cac_table['new_users'].fillna(0)

In [ ]:
# calculate CAC
cac_table['cac'] = np.where(
    cac_table['new_users'] > 0,
    cac_table['spend'] / cac_table['new_users'],
    np.nan
)

#### 4. Final datasets for SQL

In [ ]:
# Convert Period[M] columns to timestamp (first day of month) for correct mapping in PostgreSQL

# marketing_spend
marketing_spend['date'] = marketing_spend['date'].dt.to_timestamp()

# new_users
new_users['signup_date'] = new_users['signup_date'].dt.to_timestamp()

# cac_table
cac_table['date'] = cac_table['date'].dt.to_timestamp()
cac_table['signup_date'] = cac_table['signup_date'].dt.to_timestamp()


In [ ]:
accounts.to_csv('data/data_clean/accounts_clean.csv', index=False)
subscriptions.to_csv('data/data_clean/subscriptions_clean.csv', index=False)

marketing_spend.to_csv('data/data_clean/marketing_spend.csv', index=False)
new_users.to_csv('data/data_clean/new_users.csv', index=False)
cac_table.to_csv('data/data_clean/cac_table.csv', index=False)

feature_usage.to_csv('data/data_clean/feature_usage_clean.csv', index=False)
support_tickets.to_csv('data/data_clean/support_tickets_clean.csv', index=False)
churn_events.to_csv('data/data_clean/churn_events_clean.csv', index=False)

Cleaned datasets are saved for further processing in SQL.

#### Summary

- Data has been cleaned and standardized
- Marketing channel information has been prepared
- Synthetic marketing spend data has been generated
- Datasets are fully prepared for SQL-based modeling and unit economics analysis (LTV, CAC, churn)

*Limitations*:

- Marketing channels are approximated using referral_source
- Marketing spend is synthetic and should be interpreted directionally
- CAC values are simulated and used for analytical demonstration purposes